<a href="https://colab.research.google.com/github/GitSid-glitch/ML_Colab/blob/main/Copy_of_airbnb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Notebook Overview

* **Goal:** Build an end-to-end regression workflow on synthetic Airbnb-style data using sklearn pipelines.

**Workflow:**
1. Import required libraries
2. Create synthetic dataset
3. Define feature groups and split data
4. Build preprocessing with `ColumnTransformer`
5. Train multiple regression models and compare performance

**Evaluation Metrics:**
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R2 score

**Documentation:**
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html) - Complete sklearn guide
- [Pandas documentation](https://pandas.pydata.org/docs/) - Data handling with DataFrames
- [NumPy documentation](https://numpy.org/doc/stable/) - Numerical computing

## 1. Import Libraries

* **Import Libraries:** This cell imports all libraries required for data generation, preprocessing, model training, and evaluation.

**Hints:**
- Use `numpy` for random number generation and numerical operations
- Use `pandas` to create and inspect tabular data
- Import sklearn modules for preprocessing, pipelines, models, and metrics

**What each function/class does:**

| Function/Class | Description |
|----------------|-------------|
| `np.random.default_rng(42)` | Creates a reproducible random number generator |
| `ColumnTransformer` | Applies different preprocessing steps to different column groups |
| `Pipeline` | Chains preprocessing and model training into one object |
| `SimpleImputer` | Fills missing values using a chosen strategy |
| `OneHotEncoder` | Converts categorical values into numeric indicator columns |
| `train_test_split` | Splits data into train and test sets |
| `mean_absolute_error`, `mean_squared_error`, `r2_score` | Regression evaluation metrics |

**Documentation:**
- [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor

## 2. Create Synthetic Dataset

* **Generate Data:** This cell creates an Airbnb-like dataset with binary, numeric, and categorical features, then constructs a synthetic target variable.

**Hints:**
- Use `rng.choice()` for sampling city names with probabilities
- Use `rng.normal()` and `rng.integers()` for numeric feature generation
- Introduce missing values manually to simulate real data
- Build target `1_year_revenue` using a weighted formula and random noise

**What each function does:**

| Function | Description |
|----------|-------------|
| `rng.choice(values, size, p=...)` | Randomly samples values with custom probabilities |
| `rng.normal(mean, std, size)` | Generates normally distributed values |
| `rng.integers(low, high, size)` | Generates random integer values |
| `np.where(condition, a, b)` | Chooses values element-wise based on condition |
| `pd.DataFrame({...})` | Creates a table from dictionary columns |
| `df.head()` | Displays first 5 rows |

**Documentation:**
- [NumPy random Generator.choice](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html)
- [NumPy where](https://numpy.org/doc/stable/reference/generated/numpy.where.html)
- [pandas.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [DataFrame.head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)

In [3]:
rng = np.random.default_rng(42)
n = 1000

cities = [
    'London', 'San Francisco', 'Tokyo', 'New York', 'Paris',
    'Berlin', 'Sydney', 'Toronto', 'Rome', 'Barcelona'
]
city_probs = [0.14, 0.08, 0.12, 0.15, 0.1, 0.08, 0.08, 0.1, 0.07, 0.08]

city = rng.choice(cities, size=n, p=city_probs)

# Binary features
instant_bookable = rng.integers(0, 2, size=n)
superhost = rng.integers(0, 2, size=n)
has_wifi = rng.integers(0, 2, size=n)

# Numeric features
base_price = rng.normal(110, 30, size=n).clip(35, 350)
seasonality = rng.normal(1.0, 0.12, size=n)
avg_nightly_price = (base_price * seasonality).round(2)
availability_next_180_days = rng.integers(15, 181, size=n).astype(float)

# Inject missing values
mask_price_nan = rng.random(n) < 0.12
mask_avail_nan = rng.random(n) < 0.1
avg_nightly_price[mask_price_nan] = np.nan
availability_next_180_days[mask_avail_nan] = np.nan

city_revenue_boost = {
    'London': 4500, 'San Francisco': 5200, 'Tokyo': 3800, 'New York': 5600,
    'Paris': 4200, 'Berlin': 2600, 'Sydney': 3400, 'Toronto': 3000,
    'Rome': 2400, 'Barcelona': 2800
}

# Target: 1 year revenue
price_filled = np.where(np.isnan(avg_nightly_price), np.nanmean(avg_nightly_price), avg_nightly_price)
avail_filled = np.where(np.isnan(availability_next_180_days), np.nanmean(availability_next_180_days), availability_next_180_days)
noise = rng.normal(0, 1800, size=n)

one_year_revenue = (
    price_filled * (avail_filled * 1.45)
    + 2500 * instant_bookable
    + 3200 * superhost
    + 1400 * has_wifi
    + np.vectorize(city_revenue_boost.get)(city)
    + noise
).round(2)

df = pd.DataFrame({
    'id_listing': np.arange(1, n + 1),
    'host_location_city': city,
    'avg-nightly-price': avg_nightly_price,
    'availability_next_180_days': availability_next_180_days,
    'instant_bookable': instant_bookable,
    'superhost': superhost,
    'has_wifi': has_wifi,
    '1_year_revenue': one_year_revenue
})

df.head()

,id_listing,host_location_city,avg-nightly-price,availability_next_180_days,instant_bookable,superhost,has_wifi,1_year_revenue
0,1,Toronto,93.73,94.0,1,1,0,22842.96
1,2,New York,107.56,128.0,0,1,1,28586.62
2,3,Rome,124.47,54.0,0,1,0,14470.44
3,4,Sydney,133.96,NaN,0,1,0,25593.84
4,5,London,76.12,NaN,1,1,1,22609.99


## 3. Define Features and Split Data

* **Feature Setup:** This cell defines target and feature groups, then splits the dataset into training and testing sets.

**Hints:**
- Keep binary, numeric, and categorical columns in separate lists
- Build `X` using feature columns and `y` using target column
- Use a fixed `random_state` for reproducible train-test split

**What each function does:**

| Function | Description |
|----------|-------------|
| `df[columns]` | Selects one or more columns from DataFrame |
| `train_test_split(X, y, test_size, random_state)` | Splits features and target into train and test subsets |
| `X_train.shape` | Returns dimensions of training feature matrix |
| `X_test.shape` | Returns dimensions of testing feature matrix |

**Documentation:**
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [Pandas indexing](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [NumPy/Pandas shape attribute](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html)

In [5]:
target = df['1_year_revenue']  # hint: revenue column name

binary = ['instant_bookable', 'superhost', 'has_wifi']
numeric = ['avg-nightly-price', 'availability_next_180_days']
categorical = ['host_location_city']

X = df[binary + numeric + categorical].copy()  # hint: list variable with categorical columns
y = df["1_year_revenue"].copy()  # hint: target variable name

X_train, X_test, y_train, y_test = train_test_split(  # hint: sklearn split function
    X, y, test_size=0.2, random_state=42  # hint: test fraction
)

print('Train shape:', X_train.shape)  # hint: attribute for dimensions
print('Test shape :', X_test.shape)

Train shape: (800, 6)
Test shape : (200, 6)


## 4. Build Preprocessing Pipeline

* **Preprocessing:** This cell builds a sklearn `ColumnTransformer` so each feature type is processed appropriately.

**Hints:**
- Pass binary columns directly using `passthrough`
- Impute numeric missing values with `mean`
- Impute categorical missing values with `most_frequent`
- One-hot encode categorical columns for model compatibility

**What each function/class does:**

| Function/Class | Description |
|----------------|-------------|
| `ColumnTransformer([...])` | Combines multiple column-specific transformations |
| `('binary', 'passthrough', binary)` | Keeps binary columns unchanged |
| `Pipeline([...])` | Applies sequential steps for a column group |
| `SimpleImputer(strategy='mean')` | Replaces numeric missing values with column mean |
| `SimpleImputer(strategy='most_frequent')` | Replaces categorical missing values with mode |
| `OneHotEncoder(handle_unknown='ignore')` | Converts categories to one-hot vectors and ignores unseen test categories |

**Documentation:**
- [ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data)
- [SimpleImputer](https://scikit-learn.org/stable/modules/impute.html)
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

In [6]:
preprocess = ColumnTransformer(  # hint: column-wise preprocessing class
    transformers=[
        ('binary', 'passthrough', binary),  # hint: keep columns unchanged
        ('numeric', Pipeline([  # hint: sequential transformer container
            ('imputer', SimpleImputer(strategy='mean'))  # hint: imputer with mean strategy
        ]), numeric),
        ('categorical', Pipeline([  # hint: sequential transformer container
            ('imputer', SimpleImputer(strategy='most_frequent')),  # hint: most frequent strategy
            ('encoder', OneHotEncoder(handle_unknown='ignore'))  # hint: ignore unseen categories
        ]), categorical)
    ]
)

preprocess

ColumnTransformer(transformers=[('binary', 'passthrough',
                                 ['instant_bookable', 'superhost', 'has_wifi']),
                                ('numeric',
                                 Pipeline(steps=[('imputer', SimpleImputer())]),
                                 ['avg-nightly-price',
                                  'availability_next_180_days']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['host_location_city'])])

## 5. Train Models and Compare Performance

* **Model Comparison:** This cell trains four regressors using the same preprocessing pipeline and compares their performance on test data.

**Hints:**
- Wrap each model with a single sklearn `Pipeline` using shared preprocessing
- Fit on training data and predict on test data
- Compare models using MAE, RMSE, and R2
- Sort results by `R2` to identify best-performing model quickly

**What each function/class does:**

| Function/Class | Description |
|----------------|-------------|
| `LinearRegression()` | Baseline linear model for regression |
| `DecisionTreeRegressor(...)` | Tree-based model that captures non-linear patterns |
| `BaggingRegressor(...)` | Ensemble of bootstrapped base models to reduce variance |
| `GradientBoostingRegressor(...)` | Boosting ensemble that sequentially improves weak learners |
| `Pipeline([('preprocess', ...), ('model', ...)])` | Ensures identical preprocessing for each model |
| `mean_absolute_error(y_true, y_pred)` | Average absolute prediction error |
| `np.sqrt(mean_squared_error(...))` | Root mean squared error |
| `r2_score(y_true, y_pred)` | Proportion of variance explained |

**Documentation:**
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [BaggingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingRegressor.html)
- [GradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)
- [Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)

In [7]:
# 5) Train models and compare performance
models = {
    'Linear Regression': LinearRegression(),  # hint: linear regression model class
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),  # hint: decision tree regressor class
    'Bagging Regressor': BaggingRegressor(n_estimators=100, random_state=42),  # hint: bagging regressor and estimator count
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)  # hint: gradient boosting regressor class
}

results = []
for name, estimator in models.items():  # hint: iterate over dict key-value pairs
    model_pipeline = Pipeline([  # hint: sklearn pipeline class
        ('preprocess', preprocess),
        ('model', estimator)
    ])

    model_pipeline.fit(X_train, y_train)  # hint: training method
    pred = model_pipeline.predict(X_test)  # hint: inference method

    results.append({
        'model': name,
        'MAE': round(mean_absolute_error(y_test, pred), 2),  # hint: mean absolute error function
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 2),  # hint: mean squared error function
        'R2': round(r2_score(y_test, pred), 4)  # hint: r2 metric function
    })

results_df = pd.DataFrame(results).sort_values(by='R2', ascending=False).reset_index(drop=True)  # hint: metric column used for ranking
results_df

,model,MAE,RMSE,R2
0,Gradient Boosting,1811.85,2244.32,0.9329
1,Bagging Regressor,2208.97,2735.25,0.9003
2,Linear Regression,2075.87,2753.90,0.8989
3,Decision Tree,2776.25,3542.06,0.8328
